# Tarea 03: Notebooks bajo control de versiones

**Semana 3** · Entrega: antes de la clase de la Semana 4

## Objetivos de la tarea

1. Manipular un notebook como lo que es por dentro: un diccionario de
   Python.
2. Implementar a mano lo que hace `nbstripout`, sin modificar el notebook
   original.
3. Medir qué cambió **de verdad** entre dos versiones de un notebook.


In [23]:
import copy

## Ejercicio 1 (caso base)

Escribe la función `limpiar_salidas(notebook)`, que hace a mano lo que el
filtro `nbstripout` hace automáticamente: recibe un notebook —el
diccionario que vimos en clase, con su lista de `cells`— y devuelve **uno
nuevo, sin salidas**.

Reglas exactas, que son las que califica `tests/test_tarea.py`:

- Devuelve un notebook **nuevo**. El que recibe no se modifica: después de
  llamarla, el original conserva sus salidas y sus `execution_count`. Es
  la misma idea de inmutabilidad de la semana 2, ahora sobre un
  diccionario; `copy.deepcopy` es la forma más directa de lograrlo.
- En cada celda con `cell_type == "code"`: `outputs` queda como lista
  vacía (`[]`) y `execution_count` queda en `None`.
- Las celdas de markdown se quedan **exactamente igual**. Ojo con esto: no
  tienen las llaves `outputs` ni `execution_count`, y no hay que
  agregárselas.
- Todo lo demás del notebook (`metadata`, `nbformat`, el `source` de cada
  celda) queda intacto.


In [24]:
import copy

def limpiar_salidas(notebook):
    """Devuelve un notebook nuevo, sin salidas, sin modificar el original."""
    notebook_nuevo = copy.deepcopy(notebook)
    
    for celda in notebook_nuevo.get("cells", []):
        if celda.get("cell_type") == "code":
            celda["outputs"] = []
            celda["execution_count"] = None
            
    return notebook_nuevo

## Ejercicio 2 (caso límite)

Ahora la pregunta que Git no sabe contestar sobre un `.ipynb`: entre dos
versiones de un notebook, **¿qué cambió de verdad?**

Escribe `celdas_con_cambio_real(anterior, nuevo)`, que devuelve la lista
**ordenada** de los índices de las celdas cuya fuente cambió. Reusa
`limpiar_salidas` del ejercicio 1 —esa es la idea: comparar las versiones
limpias— y no vuelvas a escribirla, la celda de arriba ya la definió.

- Una salida distinta **no** es un cambio: si dos celdas tienen el mismo
  `source` pero distintos `outputs` o `execution_count`, ese índice no
  aparece en la lista.
- El `source` de una celda puede venir como **lista de líneas** o como una
  sola **cadena**, y las dos formas representan el mismo contenido:
  `["import copy"]` y `"import copy"` son iguales para esta función.
- Si los dos notebooks tienen distinto número de celdas, los índices que
  solo existen en uno de ellos **sí** cuentan como cambio. Por ejemplo, si
  `anterior` tiene 2 celdas y `nuevo` tiene 3, el índice `2` va en la
  lista.

Ejemplo de lo que debe pasar: si corres el mismo notebook dos veces sin
tocar el código, la respuesta es `[]` — aunque el archivo pese distinto y
`git diff` marque miles de caracteres.


In [26]:
import copy
def celdas_con_cambio_real(anterior, nuevo):
    """Índices de las celdas cuya fuente cambió entre las dos versiones."""
    # 1. Limpiamos el "ruido" de las salidas usando tu función anterior
    ant_limpio = limpiar_salidas(anterior)
    nue_limpio = limpiar_salidas(nuevo)
    
    celdas_ant = ant_limpio.get("cells", [])
    celdas_nue = nue_limpio.get("cells", [])
    
    cambios = []
    max_celdas = max(len(celdas_ant), len(celdas_nue))
    
    for i in range(max_celdas):
        # 2. Caso de longitud distinta: si el índice ya no existe en alguno, cambió
        if i >= len(celdas_ant) or i >= len(celdas_nue):
            cambios.append(i)
        else:
            # Hacemos copias superficiales para no modificar los diccionarios originales
            celda_a = celdas_ant[i].copy()
            celda_n = celdas_nue[i].copy()
            
            # 3. Normalizamos el "source": si viene en lista, lo fusionamos a cadena
            fuente_a = celda_a.get("source", "")
            celda_a["source"] = "".join(fuente_a) if isinstance(fuente_a, list) else fuente_a
            
            fuente_n = celda_n.get("source", "")
            celda_n["source"] = "".join(fuente_n) if isinstance(fuente_n, list) else fuente_n
            
            # Comparamos las celdas limpias y con el formato de texto unificado
            if celda_a != celda_n:
                cambios.append(i)
                
    return cambios

## Resumen

Esta semana la entrega son **dos cosas**:

1. **Tu Pull Request al roster**, dentro de tu propio fork: tu fila en
   `docs/roster.md` con tu usuario de GitHub y el enlace a tu fork. Es tu
   primer PR del curso.
2. **Este notebook resuelto**, entregado también por Pull Request dentro de
   tu fork. Los tests de `tests/test_tarea.py` lo califican solos y dejan
   un check ✅ o ❌ en tu PR.

El paso a paso de las dos entregas —fork, `upstream`, rama, PR y el
selector *base repository* que hay que revisar— está en
[`docs/git-guia.md`](../../docs/git-guia.md).

Antes de entregar, corre el notebook completo desde un kernel limpio
(`Kernel → Restart & Run All`) y confirma que llega hasta el final sin
errores. Un notebook que no ejecuta completo no se puede calificar.
